# Whisper Speech-to-Text demo (Colab)

Transcribes speech with [openai/whisper-large-v3-turbo](https://huggingface.co/openai/whisper-large-v3-turbo) (English and Slovak), and shows its built-in speech-to-English translation mode — no separate MT model needed. Caches the model to Google Drive, same as the OmniVoice notebook, so you don't re-download it every session.

In [ ]:
# Colab already ships a working torch/CUDA pair, so only install what's missing.
!pip install -q -U transformers accelerate soundfile pydub torchaudio datasets

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_CACHE_DIR = "/content/drive/MyDrive/whisper_cache"
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = DRIVE_CACHE_DIR

In [ ]:
import torch
from transformers import pipeline

ASR_MODEL_ID = "openai/whisper-large-v3-turbo"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device.startswith("cuda") else torch.float32
pipeline_device = 0 if device.startswith("cuda") else -1

try:
    asr = pipeline(
        "automatic-speech-recognition",
        model=ASR_MODEL_ID,
        torch_dtype=dtype,
        device=pipeline_device,
    )
except Exception as e:
    print(f"Drive cache unavailable ({e!r}); falling back to a fresh download.")
    os.environ.pop("HF_HOME", None)
    asr = pipeline(
        "automatic-speech-recognition",
        model=ASR_MODEL_ID,
        torch_dtype=dtype,
        device=pipeline_device,
    )

## Sanity check

Transcribes one public sample clip with a known transcript, to confirm the model works before moving to the mic.

In [ ]:
from datasets import load_dataset

sample = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")[0]
result = asr(sample["audio"])

print("Reference:  ", sample["text"])
print("Transcribed:", result["text"])

In [ ]:
from google.colab.output import eval_js
from IPython.display import Javascript, Audio, display
from base64 import b64decode
from pydub import AudioSegment
import io
import json
import numpy as np
import torchaudio

_RECORD_JS = """
async function record(ms, prompt) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });

  const div = document.createElement('div');
  div.innerHTML = `<p>Click, then <b>${prompt}</b> (auto-stops after ${ms / 1000}s)</p><button>🎤 Start</button>`;
  document.body.appendChild(div);
  const btn = div.querySelector('button');
  await new Promise(resolve => { btn.onclick = resolve; });
  btn.textContent = `● Recording... (${ms / 1000}s)`;

  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(resolve => { recorder.onstop = resolve; });
  recorder.start();
  setTimeout(() => recorder.stop(), ms);
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  div.remove();

  const reader = new FileReader();
  reader.readAsDataURL(new Blob(chunks));
  return new Promise(resolve => { reader.onloadend = () => resolve(reader.result); });
}
"""

def record_audio(seconds=6, prompt="speak now"):
    """Record from the browser mic, returns (waveform float32 numpy, sample_rate)."""
    display(Javascript(_RECORD_JS))
    data_url = eval_js(f"record({seconds * 1000}, {json.dumps(prompt)})")
    raw = b64decode(data_url.split(",", 1)[1])

    segment = AudioSegment.from_file(io.BytesIO(raw)).set_channels(1)
    waveform = np.array(segment.get_array_of_samples()).astype(np.float32)
    waveform /= 1 << (8 * segment.sample_width - 1)
    return waveform, segment.frame_rate

def to_16k(waveform, sr):
    """Whisper expects 16kHz audio."""
    if sr == 16000:
        return waveform
    resampled = torchaudio.functional.resample(
        torch.from_numpy(waveform), orig_freq=sr, new_freq=16000
    )
    return resampled.numpy()

## Transcribe: English

Run the next cell — allow microphone access if prompted, then click **Start** and say a sentence in English. It stops automatically after 6 seconds.

In [ ]:
waveform, sr = record_audio(6, prompt="say a sentence in English")
display(Audio(waveform, rate=sr))

result = asr(
    {"array": to_16k(waveform, sr), "sampling_rate": 16000},
    generate_kwargs={"language": "english", "task": "transcribe"},
)
print("Transcribed:", result["text"])

## Transcribe: Slovak

Run the next cell — click **Start** and say a sentence in Slovak. It stops automatically after 6 seconds.

In [ ]:
waveform, sr = record_audio(6, prompt="say a sentence in Slovak")
display(Audio(waveform, rate=sr))

result = asr(
    {"array": to_16k(waveform, sr), "sampling_rate": 16000},
    generate_kwargs={"language": "slovak", "task": "transcribe"},
)
print("Transcribed:", result["text"])

## Whisper can translate too

Same model, same audio — Whisper's `task` can be `"transcribe"` (stays in the spoken language) or `"translate"` (always converts speech to English, regardless of the source language). Reuses the Slovak recording from above, so there's nothing new to record.

In [ ]:
result = asr(
    {"array": to_16k(waveform, sr), "sampling_rate": 16000},
    generate_kwargs={"language": "slovak", "task": "translate"},
)
print("Translated to English:", result["text"])

## Common ASR failure modes

Even strong models struggle with:

- **Named entities** — personal and place names, especially ones from a different language than the speech (e.g. Slovak surnames spoken in an English sentence, or vice versa).
- **Company/brand/product names** — anything newer, niche, or that sounds like an ordinary word, since the model has no domain knowledge to disambiguate from acoustics alone.
- **Numbers, dates, currency, units** — spoken-to-written conversion is inherently ambiguous ("twenty twenty-six" vs "2026"). This is exactly what the DER/DSER metrics in the `slovak-llm-audio` benchmark measure separately from word-level CER/WER.
- **Code-switching** — switching languages mid-sentence (e.g. a Slovak speaker dropping in English technical terms), since most models assume one language per utterance.
- **Low-resource languages** — bigger accuracy gaps for languages with less training data, like Slovak vs English — the reason a dedicated Slovak benchmark exists at all.
- **Homophones & rare words** — usually disambiguated by context, but that safety net is weaker for uncommon vocabulary.
- **Background noise, overlapping speech, multiple speakers** — crosstalk and noisy environments degrade accuracy sharply, and go beyond ASR into diarization.
- **Accents and dialects** — non-native or regional accents underrepresented in training data.
- **Hallucination on silence** — a well-documented Whisper-specific failure: near-silent audio can produce fabricated text instead of an empty transcript.
- **Disfluencies** — filler words, stutters, false starts; models may transcribe them literally, drop them inconsistently, or get confused by them.
- **Punctuation & capitalization** — there's no single "correct" punctuation for spoken language, so this is a common source of apparent errors that aren't really wrong words.